# 6 · A person in the loop

A node that needs a person raises `Ask`; its return type says what the answer is. Only what reads it waits, and the run ends `waiting` with every question on it. `run.answer(unit, value)` writes an answer in and `run.resume()` goes on.

In [ ]:
from typing import Annotated

from conductor import Ask, GraphRecord, NodeRecord, NodeRegistry, Param, Series
from conductor.widgets import Textarea

registry = NodeRegistry()


@registry.node("text-uppercase", category="text")
def uppercase(text: str) -> str:
    return text.upper()


@registry.node("text-split", category="text")
def split(text: str, separator: str = ",") -> Series[str]:
    return text.split(separator)


@registry.node("text-join", category="text")
def join(parts: Series[str], separator: str = "") -> str:
    return separator.join(parts)


Decision = Annotated[str, Param(title="Decision")]


@registry.node("approve", title="Approve", description="Asks a person to approve the proposal", category="review")
def approve(proposal: Annotated[str, Param(title="Proposal", widget=Textarea())] = "") -> Decision:
    """Asks a person to approve a proposal; the person's answer is the node's output."""
    raise Ask("Approve or rewrite the proposal.", default=proposal, widget=Textarea())

## One question

`shout` waits for `approve`; `other` runs in the same leg.

In [ ]:
graph = registry.load(GraphRecord(
    nodes=[
        NodeRecord(id="approve", node="approve", values={"proposal": "Ship it"}),
        NodeRecord(id="shout", node="text-uppercase"),
        NodeRecord(id="other", node="text-uppercase", values={"text": "independent"}),
    ],
    edges=[("approve.result", "shout.text")],
))

run = graph.run()

print(run.status)
for unit in run.waiting:
    print(unit.id, unit.row, unit.prompt, unit.default, unit.answer_schema)
print("ran in this leg:", sorted(node_id for node_id, outputs in run.results.items() if outputs["result"] is not None))

The next leg:

In [ ]:
answered = run.answer(run.waiting[0], "Ship it on Monday")

started = []
async for event in answered.stream():
    if event.type == "node_start":
        started.append(event.node_id)
    ended = event

print(ended.run.status)
print(ended.run.results["shout"]["result"])
print("started again:", started)

## A question per row

A node on rows asks once per row. Each waiting unit has its own id, the node and its row (`approve@1`), and an answer is given by it. A unit left out goes on waiting.

In [ ]:
graph = registry.load(GraphRecord(
    nodes=[
        NodeRecord(id="drafts", node="text-split", values={"text": "first draft,second draft"}),
        NodeRecord(id="approve", node="approve"),
        NodeRecord(id="joined", node="text-join", values={"separator": " | "}),
    ],
    edges=[("drafts.result", "approve.proposal"), ("approve.result", "joined.parts")],
))

run = graph.run()
print(run.status, [(unit.row, unit.default) for unit in run.waiting])

for unit in run.waiting:
    run = run.answer(unit, f"{unit.default}, approved")

done = run.resume()
print(done.status, done.results["joined"]["result"])

Between legs a run is JSON, so it can wait days in a host's table:

```python
saved = run.to_json()
run = graph.load_run(saved)        # in another process, against the same graph
```